# 基础示例: 构建和运行一个复合流域水文模型

## 目的
本教程是学习此框架的起点。它将通过一个具体的例子，展示如何使用`SimulationController`和多个模型组件，来构建一个由多个子流域组成的、结构化的水文模型。这代表了本框架最核心和最强大的设计思想——**组件化建模**。

我们将模拟一个由三个子流域（1, 2, 3）组成的级联系统，其拓扑结构为 `3 -> 2 -> 1`，其中`1`是最终的总出口。

此笔记本将展示如何：
1.  为每个子流域的本地“降雨-产流”过程创建一个独立的`HydrologicalModel`组件。
2.  为子流域之间的河道汇流过程创建独立的`MuskingumRouting`组件。
3.  使用`Junction`组件来合并来自上游的河道来水和本地的坡面产流。
4.  使用`SimulationController`将所有这些组件连接成一个完整的、代表真实物理过程的网络。
5.  运行整个网络模拟，并可视化最终的结果。

## 1. 环境设置

我们导入所有需要的组件。由于框架中的一些组件（如`MuskingumRouting`）不完全符合控制器要求的`BaseModelComponent`接口，我们还定义了两个**包装类（Wrapper Classes）**来解决这个问题。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from common.base_model import BaseModelComponent
from common.controller import SimulationController
from common.junction import Junction
from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting, MuskingumRouting

# --- 包装类定义 ---
class RoutingWrapper(BaseModelComponent):
    """将一个标准的汇流模块包装成一个与控制器兼容的组件。"""
    def __init__(self, name: str, routing_module):
        super().__init__(name)
        self.routing_module = routing_module
    
    def step(self, inflows: dict, dt: float):
        total_inflow = sum(inflows.values())
        self.outflow = self.routing_module.run(total_inflow)

class HydroWrapper(BaseModelComponent):
    """包装HydrologicalModel，使其可以从全局输入中选择自己的数据。"""
    def __init__(self, name: str, hydro_model: HydrologicalModel, rainfall_key: str, pet_key: str):
        super().__init__(name)
        self.hydro_model = hydro_model
        self.rainfall_key = rainfall_key
        self.pet_key = pet_key
        
    def step(self, inflows: dict, dt: float):
        rain = inflows.get(self.rainfall_key, 0)
        pet = inflows.get(self.pet_key, 0)
        self.outflow = self.hydro_model.run(rain, pet)


## 2. 加载数据和定义参数

我们加载数据，定义参数，并创建一个扁平的`global_inputs`字典，其中键是唯一的输入名称（如`rainfall_1`）。

In [ ]:
# 参数
params_zone_a = {'CN': 80}
params_zone_b = {'CN': 70}
routing_params = {'k_q': 0.8, 'k_s': 0.1}
muskingum_params = {'K': 2.0, 'x': 0.25}

# 加载数据
rainfall_df = pd.read_csv('../data/rainfall.csv', index_col='date', parse_dates=True)
pet_df = pd.read_csv('../data/pet.csv', index_col='date', parse_dates=True)
observed_flow_df = pd.read_csv('../data/observed_flow.csv', index_col='date', parse_dates=True)

# 准备扁平化的全局输入
num_steps = len(rainfall_df)
global_inputs = {
    'rainfall_1': rainfall_df['rainfall_1'].values,
    'rainfall_2': rainfall_df['rainfall_2'].values,
    'rainfall_3': rainfall_df['rainfall_3'].values,
    'pet': pet_df['pet'].values
}

## 3. 构建网络

我们使用包装类来创建组件实例。

In [ ]:
# 1. 创建所有组件实例
hydro_model3 = HydrologicalModel('hydro_model3', SCSCurveNumberModule(**params_zone_a), SimpleRouting(**routing_params))
hydro3 = HydroWrapper('Hydro3', hydro_model3, 'rainfall_3', 'pet')

route3to2 = RoutingWrapper(name='Route3to2', routing_module=MuskingumRouting(**muskingum_params))

hydro_model2 = HydrologicalModel('hydro_model2', SCSCurveNumberModule(**params_zone_b), SimpleRouting(**routing_params))
hydro2 = HydroWrapper('Hydro2', hydro_model2, 'rainfall_2', 'pet')

junction2 = Junction(name='Junction2')
route2to1 = RoutingWrapper(name='Route2to1', routing_module=MuskingumRouting(**muskingum_params))

hydro_model1 = HydrologicalModel('hydro_model1', SCSCurveNumberModule(**params_zone_b), SimpleRouting(**routing_params))
hydro1 = HydroWrapper('Hydro1', hydro_model1, 'rainfall_1', 'pet')

junction1 = Junction(name='Junction1')

all_components = [hydro3, route3to2, hydro2, junction2, route2to1, hydro1, junction1]

# 2. 在控制器中添加并连接组件
controller = SimulationController()
for comp in all_components:
    controller.add_component(comp)

controller.connect('Hydro3', 'Route3to2')
controller.connect('Route3to2', 'Junction2')
controller.connect('Hydro2', 'Junction2')
controller.connect('Junction2', 'Route2to1')
controller.connect('Route2to1', 'Junction1')
controller.connect('Hydro1', 'Junction1')

print("网络构建完成。")

## 4. 运行模拟

现在，我们使用修正后的方法来运行模拟并收集历史记录。

In [ ]:
print("运行模拟...")
history = []
history_generator = controller.run(
    num_steps=num_steps,
    dt=24*3600,
    global_inputs=global_inputs
)

for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        current_step_history[name] = {'outflow': comp.get_outflow()}
    history.append(current_step_history)

print("模拟完成。")

## 5. 可视化结果

我们提取总出口(`Junction1`)的流量，并与观测流量进行对比。

In [ ]:
simulated_flow = [h['Junction1']['outflow'] for h in history]

areas = {'Hydro3': 150, 'Hydro2': 200, 'Hydro1': 120}
dt_seconds = 24 * 3600
def mm_to_m3s(runoff_mm, area_km2):
    return (runoff_mm / 1000.0) * (area_km2 * 1e6) / dt_seconds

q_sim_m3s = []
total_area = sum(areas.values())
for i in range(num_steps):
    q_sim_m3s.append(mm_to_m3s(simulated_flow[i], total_area))

fig, ax1 = plt.subplots(figsize=(15, 7))
ax1.plot(rainfall_df.index, q_sim_m3s, 'b-', label='Simulated Flow')
ax1.plot(observed_flow_df, 'k--', label='Observed Flow')
ax1.set_xlabel('Date')
ax1.set_ylabel('Flow (m³/s)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.bar(rainfall_df.index, rainfall_df['rainfall_1'], width=0.6, color='c', alpha=0.6, label='Rainfall (at outlet)')
ax2.set_ylabel('Rainfall (mm)', color='c')
ax2.tick_params(axis='y', labelcolor='c')
ax2.invert_yaxis()

plt.title('Composite Catchment Simulation Results')
plt.tight_layout()
plt.show()